# P13B — Monitor layanan, drift, dan kinerja berlabel

Jalankan P13A lebih dahulu. Baseline bulan 1–12 dibandingkan dengan window berjalan enam bulan. Label dianggap matang setelah dua bulan dan metrik hanya dihitung atas pasangan prediksi-outcome yang tersedia; syarat minimum 72 label menghindari pembacaan kinerja dari satu bulan saja. PSI, ukuran window, dan threshold di bawah merupakan heuristik latihan, bukan ambang aman untuk operasi.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, recall_score
import matplotlib.pyplot as plt

df=pd.read_csv('Bab13_Paket/Data/bridge_monitor_stream_synthetic.csv')
assert not df.duplicated(['bridge_id','month']).any()
print('shape:',df.shape,'| assets:',df.bridge_id.nunique(),'| months:',df.month.nunique())
print('label rows still pending at end of observation window:',int((df.label_available_month_index>24).sum()))


shape: (576, 15) | assets: 24 | months: 24
label rows still pending at end of observation window: 48


In [2]:
# PSI: teaching diagnostic with quantile bins from the reference period.
base=df[df.month_index<=12]['risk_score_frozen'].to_numpy()
edges=np.unique(np.quantile(base,np.linspace(0,1,6)))
if len(edges)<3: edges=np.linspace(0,1,6)
edges[0]=-np.inf; edges[-1]=np.inf
def psi(reference,current):
    a=np.histogram(reference,bins=edges)[0]/max(len(reference),1)
    b=np.histogram(current,bins=edges)[0]/max(len(current),1)
    eps=1e-3; a=np.clip(a,eps,None); b=np.clip(b,eps,None)
    return float(np.sum((b-a)*np.log(b/a)))

records=[]
for m in sorted(df.month_index.unique()):
    g=df[(df.month_index>=max(1,m-5))&(df.month_index<=m)]
    mature=g[g.label_available_month_index<=m]
    r={'month_index':int(m),'n_predictions':len(g),'sensor_missing_frac':float(g.sensor_response.isna().mean()),
       'sensor_coverage_mean':float(g.sensor_coverage_pct.mean()),'risk_mean':float(g.risk_score_frozen.mean()),
       'risk_psi':psi(base,g.risk_score_frozen.to_numpy()),'n_mature_labels':len(mature),
       'event_rate_mature':float(mature.observed_event_synth.mean()) if len(mature) else np.nan,
       'brier_mature':float(brier_score_loss(mature.observed_event_synth,mature.risk_score_frozen)) if len(mature)>=72 else np.nan,
       'recall_at_010':float(recall_score(mature.observed_event_synth,(mature.risk_score_frozen>=.10).astype(int),zero_division=0)) if len(mature)>=72 else np.nan}
    records.append(r)
monitor=pd.DataFrame(records)
base_brier=monitor.loc[monitor.month_index<=12,'brier_mature'].mean()
monitor['alert_score_shift']=monitor.risk_psi>.20
monitor['alert_brier']=monitor.brier_mature>(base_brier+.04)
monitor['response_example']=np.select([monitor.alert_score_shift,monitor.alert_brier],
    ['Investigasi input/sensor; jangan retrain otomatis','Triase kinerja berlabel dan kelompok'],default='Monitor rutin')
print('baseline Brier (mature months):',round(base_brier,3))
print(monitor[['month_index','n_predictions','n_mature_labels','risk_psi','brier_mature','recall_at_010','alert_score_shift','alert_brier','response_example']].round(3).to_string(index=False))


baseline Brier (mature months): 0.113
 month_index  n_predictions  n_mature_labels  risk_psi  brier_mature  recall_at_010  alert_score_shift  alert_brier                                  response_example
           1             24                0     0.116           NaN            NaN              False        False                                     Monitor rutin
           2             48                0     0.061           NaN            NaN              False        False                                     Monitor rutin
           3             72               24     0.065           NaN            NaN              False        False                                     Monitor rutin
           4             96               48     0.038           NaN            NaN              False        False                                     Monitor rutin
           5            120               72     0.055         0.070          0.000              False        False                 

In [3]:
out=Path('Bab13_Paket/Data'); monitor.to_csv(out/'P13_monthly_monitor_synthetic.csv',index=False)
fig,ax=plt.subplots(3,1,figsize=(10,9),sharex=True)
ax[0].plot(monitor.month_index,monitor.risk_psi,marker='o'); ax[0].axhline(.20,color='red',ls='--'); ax[0].set_ylabel('PSI skor')
ax[1].plot(monitor.month_index,monitor.brier_mature,marker='o'); ax[1].axhline(base_brier+.04,color='red',ls='--'); ax[1].set_ylabel('Brier label matang')
ax[2].plot(monitor.month_index,monitor.sensor_coverage_mean,marker='o'); ax[2].set_ylabel('Cakupan sensor (%)'); ax[2].set_xlabel('Bulan prediksi')
fig.suptitle('Monitor sintesis: sinyal distribusi, outcome matang, cakupan')
fig.tight_layout(); plt.show()


## Interpretasi dan batas

PSI meningkat menunjukkan perubahan distribusi skor relatif terhadap baseline/bins yang dipilih; ia bukan ukuran error. Brier dan recall hanya mencakup label yang sudah matang, sehingga bandingkan juga ukuran dan komposisi sampel. Sensor-shift dapat memicu alarm distribusi lebih awal; event shift baru tampak ketika outcome tersedia. Perubahan apa pun mengundang pemeriksaan dan keputusan pemilik, bukan pembaruan model otomatis.


In [4]:
print('Saved:',out/'P13_monthly_monitor_synthetic.csv')
print('Heuristic thresholds used only for exercise: PSI > 0.20; Brier > baseline + 0.04; min mature labels = 72; score threshold = 0.10; rolling window = 6 months')


Saved: Bab13_Paket\Data\P13_monthly_monitor_synthetic.csv
Heuristic thresholds used only for exercise: PSI > 0.20; Brier > baseline + 0.04; min mature labels = 72; score threshold = 0.10; rolling window = 6 months
